# `test_trader` — offline signals vs CSV mids

Loads aligned **NVDA** / **NVDA_DUAL** mids from `data/csv/`, replays :meth:`Trader.replay_dual_listing`, and plots where the dual-listing *entry-style* signals fire (not full live position logic).

Run from **repo root** with `PYTHONPATH=.` so `import trader` works.

In [ ]:
from pathlib import Path
import sys

REPO = None
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "trader.py").is_file():
        REPO = _p
        break
if REPO is None:
    raise RuntimeError("Run from repo root (need trader.py)")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

CSV_DIR = REPO / "data" / "csv"
print("REPO", REPO)
print("CSV_DIR", CSV_DIR)

In [ ]:
import pandas as pd
import numpy as np

def load_mid_series(csv_dir: Path, inst: str, tail: int | None = 5000):
    path = csv_dir / f"{inst}_strategy_market_data.csv"
    df = pd.read_csv(path, usecols=["timestamp", "mid"], encoding="utf-8-sig")
    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df["mid"] = pd.to_numeric(df["mid"], errors="coerce")
    df = df.dropna(subset=["timestamp", "mid"])
    if tail is not None:
        df = df.tail(int(tail))
    return df

main_id, dual_id = "NVDA", "NVDA_DUAL"
dm = load_mid_series(CSV_DIR, main_id)
dd = load_mid_series(CSV_DIR, dual_id)
wide = dm.merge(dd, on="timestamp", how="inner", suffixes=("_m", "_d"))
wide = wide.sort_values("timestamp").reset_index(drop=True)
main_mids = wide["mid_m"].to_numpy(dtype=float)
dual_mids = wide["mid_d"].to_numpy(dtype=float)
print("aligned rows", len(wide))
wide.head()

In [ ]:
from trader import Trader

events = Trader.replay_dual_listing(main_mids, dual_mids)
print("n_events", len(events))
for row in events[:12]:
    print(row)
if len(events) > 12:
    print("...")

In [ ]:
import matplotlib.pyplot as plt

ratio = main_mids / dual_mids
x_ev = np.arange(len(ratio), dtype=float)

kind_style = {
    "short_spread": dict(color="red", marker="v", label="Short spread (MAIN>DUAL)"),
    "long_spread": dict(color="green", marker="^", label="Long spread (DUAL>MAIN)"),
    "lag_buy": dict(color="lime", marker="+", s=80, label="Lag buy DUAL"),
    "lag_sell": dict(color="orange", marker="x", s=60, label="Lag sell DUAL"),
}

plt.figure(figsize=(11, 5), dpi=120)
plt.plot(x_ev, ratio, color="tab:blue", linewidth=1.0, label="NVDA / NVDA_DUAL")

seen = set()
for ev in events:
    k = ev["kind"]
    st = kind_style[k]
    lab = st["label"] if k not in seen else None
    seen.add(k)
    kwargs = {a: st[a] for a in st if a != "label"}
    plt.scatter(ev["i"], ratio[int(ev["i"])], label=lab, zorder=5, **kwargs)

plt.xlabel("Event number (aligned CSV rows)")
plt.ylabel("Ratio")
plt.title("Offline dual-listing signals (Trader.replay_dual_listing)")
plt.legend(loc="best", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()